# Behavioral Credit Risk Scoring System - Dask ML Pipeline

This notebook implements a scalable credit risk scoring system using:
- **Spark**: Data ingestion, cleaning, feature engineering, and dataset creation
- **Dask**: Distributed model training, hyperparameter tuning, and evaluation

## Architecture

```
Spark ──► Parquet ──► Dask ──► Model Training ──► Evaluation ──► Scoring
```

## Usage

1. **Full Pipeline**: Run all cells from Module 1 to Module 16
2. **Skip Data Processing**: Set `SKIP_DATA = True` in Module 1 to use existing data
3. **Individual Modules**: Each module can be run independently
4. **Custom Sampling**: Adjust `SAMPLE_FRACTION` for Dask data sampling

## Key Features

- ✅ No full dataset `.toPandas()` calls
- ✅ Minimal `.compute()` calls
- ✅ Distributed training with Dask
- ✅ Memory-efficient CatBoost
- ✅ Optimized stacking ensemble
- ✅ Production-ready pipeline

## Requirements

```bash
pip install -r requirements.txt
```

In [1]:
# =============================================================================
# CONFIGURATION CELL - Set parameters before running
# =============================================================================

# ──── Pipeline Configuration ────

# Set to True to skip all data processing and use existing parquet files
# This is useful when you've already run the full pipeline once
SKIP_DATA = False  # ← Set to True to use existing data

# Set to True to force re-processing even if data exists
FORCE = False

# ──── Dask Configuration ────

# Number of Dask workers for distributed training
N_WORKERS = 2

# Threads per Dask worker
THREADS_PER_WORKER = 1

# Fraction of data to sample for Dask modeling (0.0-1.0)
# Lower values = less memory usage, faster training
SAMPLE_FRACTION = 0.1  # 20% of data for training

# ──── Tuning Configuration ────

# Number of Optuna trials for hyperparameter tuning
N_TRIALS = 20

# Delinquency threshold for target creation (30, 60, or 90 days)
DEFAULT_THRESHOLD = 90

# ──── Module Selection ────

# Set to True to run the full pipeline
RUN_FULL_PIPELINE = True

# If RUN_FULL_PIPELINE is False, set the specific module to run
# Available modules: ingestion, cleaning, feature_engineering, target_creation,
#                   dataset_creation, tuning, training, evaluation, calibration,
#                   scoring, visualization
SELECTED_MODULE = 'full'  # ← Change to specific module if not running full

In [2]:
# =============================================================================
# IMPORTS & SETUP
# =============================================================================

import os
import sys
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import dask.dataframe as dd
from dask.distributed import Client
from pyspark.sql import SparkSession
from datetime import datetime
import pickle
import json
import matplotlib.pyplot as plt
import seaborn as sns
import logging

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Import configuration
from config.config import config
paths = config['paths']
model_config = config['model']
feature_config = config['features']

# Create directories
for d in [paths.bronze_dir, paths.silver_dir, paths.features_dir, 
          paths.models_dir, paths.results_dir, paths.eda_dir]:
    os.makedirs(d, exist_ok=True)

print("=" * 80)
print("BEHAVIORAL CREDIT RISK SCORING SYSTEM (Dask ML Pipeline)")
print(f"Started at: {datetime.now()}")
print("=" * 80)
print(f"Data Directory: {paths.data_dir}")
print(f"Bronze Directory: {paths.bronze_dir}")
print(f"Silver Directory: {paths.silver_dir}")
print(f"Features Directory: {paths.features_dir}")
print(f"Models Directory: {paths.models_dir}")
print(f"Results Directory: {paths.results_dir}")
print("=" * 80)
print(f"SKIP_DATA: {SKIP_DATA}")
print(f"FORCE: {FORCE}")
print(f"SAMPLE_FRACTION: {SAMPLE_FRACTION}")
print(f"N_WORKERS: {N_WORKERS}")
print(f"N_TRIALS: {N_TRIALS}")
print(f"DEFAULT_THRESHOLD: {DEFAULT_THRESHOLD}")
print("=" * 80)

BEHAVIORAL CREDIT RISK SCORING SYSTEM (Dask ML Pipeline)
Started at: 2026-07-11 02:10:01.194120
Data Directory: D:/Projects/credit_risk_scoring/data
Bronze Directory: D:/Projects/credit_risk_scoring/data/bronze
Silver Directory: D:/Projects/credit_risk_scoring/data/silver
Features Directory: D:/Projects/credit_risk_scoring/data/features
Models Directory: D:/Projects/credit_risk_scoring/models
Results Directory: D:/Projects/credit_risk_scoring/results
SKIP_DATA: False
FORCE: False
SAMPLE_FRACTION: 0.1
N_WORKERS: 2
N_TRIALS: 20
DEFAULT_THRESHOLD: 90


In [ ]:
# =============================================================================
# MODULE 1: SPARK SESSION CREATION & DASK CLIENT
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 1: SPARK SESSION & DASK CLIENT")
print("=" * 80)

from data_ingestion.data_ingestion import SFLLDDataIngestion, create_spark_session

# Create Spark session
spark = create_spark_session()
print(f"Spark Session created: {spark.version}")

# Create Dask client with less aggressive settings
print("\nInitializing Dask client (reduced resources)...")
try:
    # Use fewer workers and less memory per worker
    dask_client = Client(
        n_workers=2,  # Reduced from 4 to 2
        threads_per_worker=1,  # Reduced from 2 to 1
        memory_limit='4GB',  # Limit memory per worker
        processes=True,
        local_directory='D:/spark/spark-temp'  # Use local disk for spill
    )
    print(f"Dask client initialized with 2 workers")
    print(f"Dask dashboard: {dask_client.dashboard_link}")
except Exception as e:
    print(f"Could not initialize Dask client: {e}")
    print("Using single-threaded mode.")
    dask_client = None

# Initialize Spark components
ingestor = SFLLDDataIngestion(spark)


MODULE 1: SPARK SESSION & DASK CLIENT


2026-07-11 02:10:05,702 - data_ingestion.data_ingestion - INFO - Spark Session created: 3.5.1


fs.defaultFS = file:///
fs.default.name = file:///
Spark Session created: 3.5.1

Initializing Dask client (reduced resources)...
Dask client initialized with 2 workers
Dask dashboard: http://127.0.0.1:53114/status


2026-07-11 02:12:11,914 - distributed.nanny.memory - WARNING - Worker tcp://127.0.0.1:53131 (pid=41436) exceeded 95% memory budget. Restarting...
2026-07-11 02:12:11,931 - distributed.scheduler - WARNING - Removing worker 'tcp://127.0.0.1:53131' caused the cluster to lose already computed task(s), which will be recomputed elsewhere: {('repartitionsize-b321bdde3e5722c41de4068695c66c31', 16), ('repartitionsize-6265b2461fba028928cd9d4194907ba0', 11), ('repartitionsize-6265b2461fba028928cd9d4194907ba0', 5), ('repartitionsize-6265b2461fba028928cd9d4194907ba0', 14), ('repartitionsize-6265b2461fba028928cd9d4194907ba0', 20), ('repartitionsize-b321bdde3e5722c41de4068695c66c31', 22), ('repartitionsize-6265b2461fba028928cd9d4194907ba0', 17), ('repartitionsize-6265b2461fba028928cd9d4194907ba0', 23), ('repartitionsize-b321bdde3e5722c41de4068695c66c31', 25), ('repartitionsize-b321bdde3e5722c41de4068695c66c31', 31), ('repartitionsize-6265b2461fba028928cd9d4194907ba0', 26), ('repartitionsize-6265b2461

In [4]:
# =============================================================================
# MODULE 2: DATA INGESTION
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 2: DATA INGESTION")
print("=" * 80)

if SKIP_DATA:
    print("SKIPPING ingestion (SKIP_DATA is enabled)")
else:
    print("\nLoading bronze data...")
    
    # Check if bronze data exists
    bronze_exists = False
    try:
        spark.read.parquet(paths.origination_bronze)
        spark.read.parquet(paths.performance_bronze)
        bronze_exists = True
        print("Bronze data already exists.")
    except:
        pass

    if bronze_exists and not FORCE:
        print("Loading existing bronze data...")
        origination_df = spark.read.parquet(paths.origination_bronze)
        performance_df = spark.read.parquet(paths.performance_bronze)
        print(f"Origination: {origination_df.count():,} loans")
        print(f"Performance: {performance_df.count():,} records")
    else:
        print("Ingesting data from scratch...")
        raw_dir = paths.raw_dir
        years = list(range(1999, 2013))

        # Ingest origination
        print("Ingesting origination data...")
        origination_df = ingestor.ingest_all_years(
            raw_dir=raw_dir,
            years=years,
            bronze_dir=paths.bronze_dir,
            file_prefix="sample",
            data_type="origination"
        )

        # Ingest performance
        print("Ingesting performance data...")
        performance_df = ingestor.ingest_all_years(
            raw_dir=raw_dir,
            years=years,
            bronze_dir=paths.bronze_dir,
            file_prefix="sample",
            data_type="performance"
        )

        print(f"Origination: {origination_df.count():,} loans")
        print(f"Performance: {performance_df.count():,} records")

print("\n✅ Module 2 completed.")


MODULE 2: DATA INGESTION

Loading bronze data...
Bronze data already exists.
Loading existing bronze data...
Origination: 700,000 loans
Performance: 43,612,276 records

✅ Module 2 completed.


In [5]:
# =============================================================================
# MODULE 3: DATA CLEANING
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 3: DATA CLEANING")
print("=" * 80)

from preprocessing.cleaning import SFLLDDataCleaner

if SKIP_DATA:
    print("SKIPPING cleaning (SKIP_DATA is enabled)")
else:
    # Initialize cleaner
    cleaner = SFLLDDataCleaner(spark)

    # Check if silver data exists
    silver_exists = False
    try:
        spark.read.parquet(f"{paths.silver_dir}/origination_cleaned.parquet")
        spark.read.parquet(f"{paths.silver_dir}/performance_cleaned.parquet")
        silver_exists = True
        print("Silver data already exists.")
    except:
        pass

    if silver_exists and not FORCE:
        print("\nLoading existing cleaned data...")
        orig_cleaned = spark.read.parquet(f"{paths.silver_dir}/origination_cleaned.parquet")
        perf_cleaned = spark.read.parquet(f"{paths.silver_dir}/performance_cleaned.parquet")
        print(f"Cleaned Origination: {orig_cleaned.count():,} loans")
        print(f"Cleaned Performance: {perf_cleaned.count():,} records")
    else:
        # Load bronze data if not already loaded
        try:
            origination_df
        except NameError:
            print("Loading bronze data...")
            origination_df = spark.read.parquet(paths.origination_bronze)
            performance_df = spark.read.parquet(paths.performance_bronze)

        print("\nCleaning data from scratch...")
        orig_cleaned, perf_cleaned = cleaner.clean_both_datasets(
            origination_df, performance_df
        )

        # Save to silver layer
        print("Saving cleaned data...")
        orig_cleaned.write.mode("overwrite") \
            .option("compression", "snappy") \
            .parquet(f"{paths.silver_dir}/origination_cleaned.parquet")

        perf_cleaned.write.mode("overwrite") \
            .option("compression", "snappy") \
            .parquet(f"{paths.silver_dir}/performance_cleaned.parquet")

        print(f"Cleaned Origination: {orig_cleaned.count():,} loans")
        print(f"Cleaned Performance: {perf_cleaned.count():,} records")

print("\n✅ Module 3 completed.")


MODULE 3: DATA CLEANING
Silver data already exists.

Loading existing cleaned data...
Cleaned Origination: 700,000 loans
Cleaned Performance: 43,612,276 records

✅ Module 3 completed.


In [6]:
# =============================================================================
# MODULE 4: FEATURE ENGINEERING
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 4: FEATURE ENGINEERING")
print("=" * 80)

from features.behavioral_features import BehavioralFeatureEngineer

if SKIP_DATA:
    print("SKIPPING feature engineering (SKIP_DATA is enabled)")
else:
    # Initialize feature engineer
    feature_engineer = BehavioralFeatureEngineer(spark)

    # Check if feature data exists
    feature_exists = False
    try:
        spark.read.parquet(paths.feature_dataset)
        feature_exists = True
        print("Feature data already exists.")
    except:
        pass

    if feature_exists and not FORCE:
        print("\nLoading existing feature data...")
        feature_df = spark.read.parquet(paths.feature_dataset)
        print(f"Features loaded: {feature_df.count():,} records")
        print(f"Feature count: {len(feature_df.columns)}")
    else:
        # Load cleaned data if not already loaded
        try:
            orig_cleaned
        except NameError:
            print("Loading cleaned data...")
            orig_cleaned = spark.read.parquet(f"{paths.silver_dir}/origination_cleaned.parquet")
            perf_cleaned = spark.read.parquet(f"{paths.silver_dir}/performance_cleaned.parquet")

        print("\nCreating features from scratch...")
        feature_df = feature_engineer.create_all_features(
            orig_cleaned, perf_cleaned
        )

        # Remove duplicate columns
        seen = set()
        cols_to_keep = []
        for col_name in feature_df.columns:
            if col_name not in seen:
                seen.add(col_name)
                cols_to_keep.append(col_name)
        feature_df = feature_df.select(*cols_to_keep)

        # Save feature dataset
        print("Saving feature data...")
        feature_df.write.mode("overwrite") \
            .option("compression", "snappy") \
            .parquet(paths.feature_dataset)

        print(f"Features created: {feature_df.count():,} records")
        print(f"Feature count: {len(feature_df.columns)}")

print("\n✅ Module 4 completed.")


MODULE 4: FEATURE ENGINEERING
Feature data already exists.

Loading existing feature data...
Features loaded: 43,612,276 records
Feature count: 114

✅ Module 4 completed.


In [7]:
# =============================================================================
# MODULE 5: TARGET CREATION
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 5: TARGET CREATION")
print("=" * 80)

from target.target_creation import TargetCreator

if SKIP_DATA:
    print("SKIPPING target creation (SKIP_DATA is enabled)")
else:
    # Initialize target creator
    target_creator = TargetCreator(
        spark=spark,
        default_threshold=DEFAULT_THRESHOLD,
        lookahead_months=model_config.lookahead_months
    )

    # Check if dataset with target exists
    target_exists = False
    try:
        spark.read.parquet(f"{paths.features_dir}/dataset_with_target.parquet")
        target_exists = True
        print("Dataset with target already exists.")
    except:
        pass

    if target_exists and not FORCE:
        print("\nLoading existing dataset with target...")
        dataset_df = spark.read.parquet(f"{paths.features_dir}/dataset_with_target.parquet")
        print(f"Dataset loaded: {dataset_df.count():,} records")
        target_creator.analyze_target_distribution(dataset_df)
    else:
        # Load feature data if not already loaded
        try:
            feature_df
        except NameError:
            print("Loading feature data...")
            feature_df = spark.read.parquet(paths.feature_dataset)

        print("\nCreating target from scratch...")
        dataset_df = target_creator.create_target(feature_df, threshold=DEFAULT_THRESHOLD)

        # Analyze target distribution
        print("\nTarget distribution:")
        target_creator.analyze_target_distribution(dataset_df)

        # Save dataset
        print("Saving dataset with target...")
        dataset_df.write.mode("overwrite") \
            .option("compression", "snappy") \
            .parquet(f"{paths.features_dir}/dataset_with_target.parquet")

        print(f"Dataset with target saved: {dataset_df.count():,} records")

print("\n✅ Module 5 completed.")


MODULE 5: TARGET CREATION
Dataset with target already exists.

Loading existing dataset with target...
Dataset loaded: 7,223,880 records


2026-07-11 02:11:16,336 - target.target_creation - INFO - Target distribution:
2026-07-11 02:11:16,338 - target.target_creation - INFO -   Total observations: 7,223,880
2026-07-11 02:11:16,339 - target.target_creation - INFO -   Defaults: 80,629 (1.12%)
2026-07-11 02:11:16,340 - target.target_creation - INFO -   Loans with default: 10,329



✅ Module 5 completed.


In [8]:
# =============================================================================
# MODULE 6: DATASET CREATION & SPLIT (PART 1)
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 6 : DATASET CREATION & SPLIT")
print("=" * 80)

from datasets.dataset_creation import DatasetCreator
import dask.dataframe as dd
import os

# -------------------------------------------------------------------------
# SKIP DATA
# -------------------------------------------------------------------------

if SKIP_DATA:

    print("SKIPPING Spark dataset creation...")
    print("Loading existing parquet datasets with Dask...")

    train_path = paths.train_data
    val_path = paths.val_data
    test_path = paths.test_data

    if not (
        os.path.exists(train_path)
        and os.path.exists(val_path)
        and os.path.exists(test_path)
    ):
        raise FileNotFoundError(
            "Train / Validation / Test parquet files not found.\n"
            "Run pipeline once with SKIP_DATA=False."
        )

# -------------------------------------------------------------------------
# CREATE DATASETS
# -------------------------------------------------------------------------

else:

    dataset_creator = DatasetCreator(spark)

    splits_exist = (
        os.path.exists(paths.train_data)
        and os.path.exists(paths.val_data)
        and os.path.exists(paths.test_data)
    )

    if splits_exist and not FORCE:

        print("\nExisting dataset splits found.")

        train_df = spark.read.parquet(paths.train_data)
        val_df = spark.read.parquet(paths.val_data)
        test_df = spark.read.parquet(paths.test_data)

    else:

        print("\nCreating dataset splits...")

        train_df, val_df, test_df = dataset_creator.create_dataset()

        print("\nSaving Spark train dataset...")

        (
            train_df.write
            .mode("overwrite")
            .option("compression", "snappy")
            .parquet(paths.train_data)
        )

        print("Saving Spark validation dataset...")

        (
            val_df.write
            .mode("overwrite")
            .option("compression", "snappy")
            .parquet(paths.val_data)
        )

        print("Saving Spark test dataset...")

        (
            test_df.write
            .mode("overwrite")
            .option("compression", "snappy")
            .parquet(paths.test_data)
        )

        print("Spark datasets successfully written.")

# -------------------------------------------------------------------------
# LOAD PARQUET LAZILY WITH DASK
# -------------------------------------------------------------------------

print("\nLoading datasets using Dask...")

train_dd = dd.read_parquet(
    paths.train_data,
    engine="pyarrow"
)

val_dd = dd.read_parquet(
    paths.val_data,
    engine="pyarrow"
)

test_dd = dd.read_parquet(
    paths.test_data,
    engine="pyarrow"
)

print("Datasets loaded lazily.")

# -------------------------------------------------------------------------
# REMOVE IDENTIFIER COLUMNS
# -------------------------------------------------------------------------

drop_cols = [
    "LOAN_SEQUENCE_NUMBER",
    "MONTHLY_REPORTING_PERIOD"
]

drop_cols = [
    c
    for c in drop_cols
    if c in train_dd.columns
]

print(f"\nDropping identifier columns : {drop_cols}")

# -------------------------------------------------------------------------
# CREATE FEATURE MATRICES
# -------------------------------------------------------------------------

X_train = train_dd.drop(
    columns=drop_cols + ["target"],
    errors="ignore"
)

X_val = val_dd.drop(
    columns=drop_cols + ["target"],
    errors="ignore"
)

X_test = test_dd.drop(
    columns=drop_cols + ["target"],
    errors="ignore"
)

# -------------------------------------------------------------------------
# KEEP TARGET AS DATAFRAME
# -------------------------------------------------------------------------

# IMPORTANT:
# Keep target as Dask DataFrame.
# Convert to Series only during model fitting.

y_train = train_dd[["target"]]

y_val = val_dd[["target"]]

y_test = test_dd[["target"]]

feature_names = list(X_train.columns)

data_loaded = True

print("\nDatasets prepared successfully.")
print(f"Number of features : {len(feature_names)}")


MODULE 6 : DATASET CREATION & SPLIT

Existing dataset splits found.

Loading datasets using Dask...
Datasets loaded lazily.

Dropping identifier columns : ['LOAN_SEQUENCE_NUMBER', 'MONTHLY_REPORTING_PERIOD']

Datasets prepared successfully.
Number of features : 119


In [9]:
# -------------------------------------------------------------------------
# OPTIONAL DASK SAMPLING
# -------------------------------------------------------------------------

# if SAMPLE_FRACTION < 1.0:

#     print(f"\nApplying {SAMPLE_FRACTION:.0%} sampling using Dask...")

#     # Sample features
#     X_train = X_train.sample(
#         frac=SAMPLE_FRACTION,
#         random_state=42
#     )

#     # Align target with sampled rows
#     y_train = y_train.loc[X_train.index]

#     X_val = X_val.sample(
#         frac=SAMPLE_FRACTION,
#         random_state=42
#     )

#     y_val = y_val.loc[X_val.index]

#     print("Sampling completed.")

# else:

#     print("\nUsing complete dataset.")

# -------------------------------------------------------------------------
# REPARTITION FOR BETTER PARALLELISM
# -------------------------------------------------------------------------

print("\nOptimizing Dask partitions...")

desired_partition_size = "128MB"

X_train = X_train.repartition(partition_size=desired_partition_size)
X_val = X_val.repartition(partition_size=desired_partition_size)
X_test = X_test.repartition(partition_size=desired_partition_size)

y_train = y_train.repartition(partition_size=desired_partition_size)
y_val = y_val.repartition(partition_size=desired_partition_size)
y_test = y_test.repartition(partition_size=desired_partition_size)

# -------------------------------------------------------------------------
# OPTIONAL PERSIST
# -------------------------------------------------------------------------
#
# Persist only if memory allows.
# This avoids repeated parquet reads during model fitting.
#

try:

    print("\nPersisting Dask datasets...")

    X_train = X_train.persist()
    y_train = y_train.persist()

    X_val = X_val.persist()
    y_val = y_val.persist()

    X_test = X_test.persist()
    y_test = y_test.persist()

    print("Persist successful.")

except Exception as e:

    print(f"Persist skipped ({e})")

# -------------------------------------------------------------------------
# DATA VALIDATION
# -------------------------------------------------------------------------

print("\nDataset Summary")

print("-" * 60)

print(f"Training partitions   : {X_train.npartitions}")
print(f"Validation partitions : {X_val.npartitions}")
print(f"Testing partitions    : {X_test.npartitions}")

print(f"Number of features    : {len(feature_names)}")

print("-" * 60)

# -------------------------------------------------------------------------
# LIGHTWEIGHT SHAPE INFORMATION
# -------------------------------------------------------------------------
#
# Avoid computing row counts.
#

print("\nFeature Columns")

print(feature_names)

print("\nTarget Column")

print(list(y_train.columns))

# -------------------------------------------------------------------------
# CHECK FOR MISSING TARGET
# -------------------------------------------------------------------------

if "target" not in y_train.columns:
    raise ValueError("Target column missing.")

# -------------------------------------------------------------------------
# SAVE DASK METADATA (OPTIONAL)
# -------------------------------------------------------------------------

metadata = {

    "feature_count": len(feature_names),

    "feature_names": feature_names,

    "sample_fraction": SAMPLE_FRACTION,

    "train_partitions": X_train.npartitions,

    "validation_partitions": X_val.npartitions,

    "test_partitions": X_test.npartitions

}

metadata_path = os.path.join(
    paths.results_dir,
    "dataset_metadata.json"
)

with open(metadata_path, "w") as f:

    json.dump(metadata, f, indent=4)

print(f"\nDataset metadata saved to:\n{metadata_path}")

# -------------------------------------------------------------------------
# READY FOR MODEL TRAINING
# -------------------------------------------------------------------------

print("\n" + "=" * 80)
print("MODULE 6 COMPLETED")
print("=" * 80)

print("\nDatasets ready for Dask ML.")

print(f"Training Partitions   : {X_train.npartitions}")
print(f"Validation Partitions : {X_val.npartitions}")
print(f"Testing Partitions    : {X_test.npartitions}")

print(f"Features              : {len(feature_names)}")

print("\nReady for Module 7.")

print("\n✅ Module 6 completed.")


Optimizing Dask partitions...

Persisting Dask datasets...
Persist successful.

Dataset Summary
------------------------------------------------------------
Training partitions   : 32
Validation partitions : 7
Testing partitions    : 22
Number of features    : 119
------------------------------------------------------------

Feature Columns
['CURRENT_ACTUAL_UPB', 'CURRENT_LOAN_DELINQUENCY_STATUS', 'LOAN_AGE', 'REMAINING_MONTHS_TO_LEGAL_MATURITY', 'DEFECT_SETTLEMENT_DATE', 'MODIFICATION_FLAG', 'ZERO_BALANCE_CODE', 'ZERO_BALANCE_EFFECTIVE_DATE', 'CURRENT_INTEREST_RATE', 'CURRENT_NON_INTEREST_BEARING_UPB', 'DDLPI', 'MI_RECOVERIES', 'NET_SALE_PROCEEDS', 'NON_MI_RECOVERIES', 'TOTAL_EXPENSES', 'LEGAL_COSTS', 'MAINTENANCE_AND_PRESERVATION_COSTS', 'TAXES_AND_INSURANCE', 'MISCELLANEOUS_EXPENSES', 'ACTUAL_LOSS_CALCULATION', 'CUMULATIVE_MODIFICATION_COST', 'INTEREST_RATE_STEP_INDICATOR', 'PAYMENT_DEFERRAL_FLAG', 'ELTV', 'ZERO_BALANCE_REMOVAL_UPB', 'DELINQUENT_ACCRUED_INTEREST', 'DELINQUENCY_DUE_

In [10]:
# # =============================================================================
# # MODULE 6: DATASET CREATION & SPLIT
# # =============================================================================

# print("\n" + "=" * 80)
# print("MODULE 6: DATASET CREATION & SPLIT")
# print("=" * 80)

# from datasets.dataset_creation import DatasetCreator

# if SKIP_DATA:
#     print("SKIPPING dataset creation (SKIP_DATA is enabled)")
    
#     # Try to load existing Dask data
#     print("\nLoading existing Dask data...")
#     try:
#         X_train = dd.read_parquet(f"{paths.features_dir}/dask_X_train.parquet")
#         y_train = dd.read_parquet(f"{paths.features_dir}/dask_y_train.parquet")
#         X_test = dd.read_parquet(f"{paths.features_dir}/dask_X_test.parquet")
#         y_test = dd.read_parquet(f"{paths.features_dir}/dask_y_test.parquet")
        
#         # Load validation data if available
#         X_val = None
#         y_val = None
#         if os.path.exists(f"{paths.features_dir}/dask_X_val.parquet"):
#             X_val = dd.read_parquet(f"{paths.features_dir}/dask_X_val.parquet")
#             y_val = dd.read_parquet(f"{paths.features_dir}/dask_y_val.parquet")
        
#         feature_names = X_train.columns.tolist()
#         data_loaded = True
        
#         print(f"Train shape: ({X_train.shape[0].compute():,}, {X_train.shape[1]})")
#         print(f"Test shape: ({X_test.shape[0].compute():,}, {X_test.shape[1]})")
#         if X_val is not None:
#             print(f"Validation shape: ({X_val.shape[0].compute():,}, {X_val.shape[1]})")
#     except Exception as e:
#         print(f"Could not load existing Dask data: {e}")
#         data_loaded = False
#         print("Please run the full pipeline first or set SKIP_DATA=False")
# else:
#     # Create dataset using DatasetCreator
#     dataset_creator = DatasetCreator(spark)
    
#     # Check if splits exist
#     splits_exist = False
#     try:
#         spark.read.parquet(paths.train_data)
#         spark.read.parquet(paths.val_data)
#         spark.read.parquet(paths.test_data)
#         splits_exist = True
#         print("Data splits already exist.")
#     except:
#         pass

#     if splits_exist and not FORCE:
#         print("\nLoading existing data splits...")
#         train_df = spark.read.parquet(paths.train_data)
#         val_df = spark.read.parquet(paths.val_data)
#         test_df = spark.read.parquet(paths.test_data)
        
#         print(f"Train: {train_df.count():,} records")
#         print(f"Validation: {val_df.count():,} records")
#         print(f"Test: {test_df.count():,} records")
#     else:
#         print("\nCreating dataset from scratch...")
#         train_df, val_df, test_df = dataset_creator.create_dataset()
        
#         print(f"Train: {train_df.count():,} records")
#         print(f"Validation: {val_df.count():,} records")
#         print(f"Test: {test_df.count():,} records")

#     # Convert Spark DataFrames to Dask DataFrames with sampling
#     print("\n" + "-" * 40)
#     print("Converting to Dask DataFrames...")
#     print("-" * 40)
    
#     def spark_to_dask_lazy(spark_df, sample_fraction=SAMPLE_FRACTION):
#         """Convert Spark DataFrame to Dask with sampling."""
#         total_count = spark_df.count()
        
#         # Use sampling for large datasets
#         if total_count > 100000:
#             sample_fraction = min(sample_fraction, 100000 / total_count)
#             spark_df = spark_df.sample(fraction=sample_fraction, seed=42)
        
#         # Convert to pandas (sampled data is small enough)
#         pdf = spark_df.toPandas()
        
#         # Convert to Dask with appropriate partitions
#         npartitions = max(1, len(pdf) // 50000)
#         return dd.from_pandas(pdf, npartitions=min(npartitions, 20))

#     # Drop non-feature columns
#     drop_cols = ['LOAN_SEQUENCE_NUMBER', 'MONTHLY_REPORTING_PERIOD']
    
#     # Training data
#     print(f"Sampling {SAMPLE_FRACTION*100:.1f}% of training data for Dask conversion...")
#     X_train_spark = train_df.drop(*[c for c in drop_cols if c in train_df.columns]).drop('target')
#     X_train = spark_to_dask_lazy(X_train_spark, SAMPLE_FRACTION)
#     y_train = spark_to_dask_lazy(train_df.select('target'), SAMPLE_FRACTION)['target']

#     # Validation data
#     X_val_spark = val_df.drop(*[c for c in drop_cols if c in val_df.columns]).drop('target')
#     X_val = spark_to_dask_lazy(X_val_spark, SAMPLE_FRACTION)
#     y_val = spark_to_dask_lazy(val_df.select('target'), SAMPLE_FRACTION)['target']

#     # Test data
#     X_test_spark = test_df.drop(*[c for c in drop_cols if c in test_df.columns]).drop('target')
#     X_test = spark_to_dask_lazy(X_test_spark, SAMPLE_FRACTION)
#     y_test = spark_to_dask_lazy(test_df.select('target'), SAMPLE_FRACTION)['target']

#     feature_names = X_train.columns.tolist()
#     data_loaded = True

#     print(f"Training features shape: {X_train.shape[0].compute():,}")
#     print(f"Test features shape: {X_test.shape[0].compute():,}")

#     # Save Dask DataFrames
#     print("\nSaving Dask DataFrames...")
#     dask_dir = f"{paths.features_dir}"
#     X_train.to_parquet(f"{dask_dir}/dask_X_train.parquet", write_index=False)
#     y_train.to_parquet(f"{dask_dir}/dask_y_train.parquet", write_index=False)
#     X_val.to_parquet(f"{dask_dir}/dask_X_val.parquet", write_index=False)
#     y_val.to_parquet(f"{dask_dir}/dask_y_val.parquet", write_index=False)
#     X_test.to_parquet(f"{dask_dir}/dask_X_test.parquet", write_index=False)
#     y_test.to_parquet(f"{dask_dir}/dask_y_test.parquet", write_index=False)

#     print("Dask DataFrames saved successfully")

# print("\n✅ Module 6 completed.")

In [11]:
# # =============================================================================
# # MODULE 7: LOAD DASK DATA (if not already loaded)
# # =============================================================================

# print("\n" + "=" * 80)
# print("MODULE 7: LOAD DASK DATA")
# print("=" * 80)

# # Check if data is already loaded
# try:
#     data_loaded
# except NameError:
#     data_loaded = False

# if not data_loaded:
#     print("\nLoading Dask DataFrames from disk...")
#     dask_dir = f"{paths.features_dir}"
    
#     X_train = dd.read_parquet(f"{dask_dir}/dask_X_train.parquet")
#     y_train = dd.read_parquet(f"{dask_dir}/dask_y_train.parquet")['target']
#     X_val = dd.read_parquet(f"{dask_dir}/dask_X_val.parquet")
#     y_val = dd.read_parquet(f"{dask_dir}/dask_y_val.parquet")['target']
#     X_test = dd.read_parquet(f"{dask_dir}/dask_X_test.parquet")
#     y_test = dd.read_parquet(f"{dask_dir}/dask_y_test.parquet")['target']
#     feature_names = X_train.columns.tolist()
#     data_loaded = True

# print(f"Train shape: ({X_train.shape[0].compute():,}, {X_train.shape[1]})")
# print(f"Validation shape: ({X_val.shape[0].compute():,}, {X_val.shape[1]})")
# print(f"Test shape: ({X_test.shape[0].compute():,}, {X_test.shape[1]})")

# print("\n✅ Module 7 completed.")


# =============================================================================
# MODULE 7: LOAD DASK DATA
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 7 : LOAD DASK DATA")
print("=" * 80)

import dask.dataframe as dd

try:
    data_loaded
except NameError:
    data_loaded = False

# ---------------------------------------------------------------------
# Load only if Module 6 has not already executed
# ---------------------------------------------------------------------

if not data_loaded:

    print("\nLoading datasets from Spark Parquet...")

    train_dd = dd.read_parquet(
        paths.train_data,
        engine="pyarrow"
    )

    val_dd = dd.read_parquet(
        paths.val_data,
        engine="pyarrow"
    )

    test_dd = dd.read_parquet(
        paths.test_data,
        engine="pyarrow"
    )

    drop_cols = [
        "LOAN_SEQUENCE_NUMBER",
        "MONTHLY_REPORTING_PERIOD"
    ]

    drop_cols = [
        c for c in drop_cols
        if c in train_dd.columns
    ]

    X_train = train_dd.drop(
        columns=drop_cols + ["target"],
        errors="ignore"
    )

    y_train = train_dd[["target"]]

    X_val = val_dd.drop(
        columns=drop_cols + ["target"],
        errors="ignore"
    )

    y_val = val_dd[["target"]]

    X_test = test_dd.drop(
        columns=drop_cols + ["target"],
        errors="ignore"
    )

    y_test = test_dd[["target"]]

    feature_names = list(X_train.columns)

    data_loaded = True

# ---------------------------------------------------------------------
# Display lightweight information
# ---------------------------------------------------------------------

print("\nDatasets successfully loaded.")

print(f"Training partitions   : {X_train.npartitions}")
print(f"Validation partitions : {X_val.npartitions}")
print(f"Testing partitions    : {X_test.npartitions}")

print(f"Number of features    : {len(feature_names)}")

print("\nFeature names:")

print(feature_names)

print("\n✅ Module 7 completed.")


MODULE 7 : LOAD DASK DATA

Datasets successfully loaded.
Training partitions   : 32
Validation partitions : 7
Testing partitions    : 22
Number of features    : 119

Feature names:
['CURRENT_ACTUAL_UPB', 'CURRENT_LOAN_DELINQUENCY_STATUS', 'LOAN_AGE', 'REMAINING_MONTHS_TO_LEGAL_MATURITY', 'DEFECT_SETTLEMENT_DATE', 'MODIFICATION_FLAG', 'ZERO_BALANCE_CODE', 'ZERO_BALANCE_EFFECTIVE_DATE', 'CURRENT_INTEREST_RATE', 'CURRENT_NON_INTEREST_BEARING_UPB', 'DDLPI', 'MI_RECOVERIES', 'NET_SALE_PROCEEDS', 'NON_MI_RECOVERIES', 'TOTAL_EXPENSES', 'LEGAL_COSTS', 'MAINTENANCE_AND_PRESERVATION_COSTS', 'TAXES_AND_INSURANCE', 'MISCELLANEOUS_EXPENSES', 'ACTUAL_LOSS_CALCULATION', 'CUMULATIVE_MODIFICATION_COST', 'INTEREST_RATE_STEP_INDICATOR', 'PAYMENT_DEFERRAL_FLAG', 'ELTV', 'ZERO_BALANCE_REMOVAL_UPB', 'DELINQUENT_ACCRUED_INTEREST', 'DELINQUENCY_DUE_TO_DISASTER', 'BORROWER_ASSISTANCE_STATUS_CODE', 'CURRENT_MONTH_MODIFICATION_COST', 'INTEREST_BEARING_UPB', 'reporting_year', 'CREDIT_SCORE', 'FIRST_PAYMENT_DATE'

In [12]:
# =============================================================================
# MODULE 8: HYPERPARAMETER TUNING
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 8: HYPERPARAMETER TUNING")
print("=" * 80)

from models.hyperparameter_tuning import HyperparameterTuner

# Check if tuned parameters exist
tuned_params_path = f"{paths.results_dir}/tuned_params.json"
if os.path.exists(tuned_params_path) and not FORCE:
    with open(tuned_params_path, 'r') as f:
        tuned_params = json.load(f)
    print(f"✅ Loaded existing tuned parameters with models: {list(tuned_params.keys())}")
else:
    print("⚠️ No existing tuned parameters found. Will create new.")
    
    # Create tuner with sampling
    tuner = HyperparameterTuner(
        n_trials=N_TRIALS,
        cv_folds=min(3, model_config.cv_folds),
        random_state=model_config.random_state,
        sample_fraction=0.1  # Use 10% for tuning
    )

    tuned_params = {}

    # Tune selected models
    models_to_tune = ['logistic', 'xgboost']

    for model_name in models_to_tune:
        print(f"\nTuning {model_name}...")
        try:
            if model_name == 'logistic':
                params = tuner.tune_logistic_regression(X_train, y_train)
            elif model_name == 'xgboost':
                params = tuner.tune_xgboost(X_train, y_train)
            elif model_name == 'lightgbm':
                params = tuner.tune_lightgbm(X_train, y_train)
            elif model_name == 'random_forest':
                params = tuner.tune_random_forest(X_train, y_train)
            elif model_name == 'catboost':
                params = tuner.tune_catboost(X_train, y_train)
            tuned_params[model_name] = params
            print(f"  ✅ {model_name} tuning completed")
        except Exception as e:
            print(f"  ❌ {model_name} tuning failed: {e}")
            tuned_params[model_name] = {}

    # Save tuned parameters
    with open(tuned_params_path, 'w') as f:
        json.dump(tuned_params, f, indent=2)

    print(f"\n✅ Parameters saved to: {tuned_params_path}")

print("\n✅ Module 8 completed.")


MODULE 8: HYPERPARAMETER TUNING
✅ Loaded existing tuned parameters with models: ['logistic', 'random_forest', 'xgboost', 'lightgbm', 'catboost']

✅ Module 8 completed.


In [ ]:
# # =============================================================================
# # MODULE 9: MODEL TRAINING (Dask Backend)
# # =============================================================================

# print("\n" + "=" * 80)
# print("MODULE 9: MODEL TRAINING (Dask Backend)")
# print("=" * 80)

# from models.logistic import LogisticRegressionModel
# from models.random_forest import RandomForestModel
# from models.xgboost_model import XGBoostModel
# from models.lightgbm_model import LightGBMModel
# from models.catboost_model import CatBoostModel
# from models.ensemble import StackingEnsemble

# # Check if models exist
# models_exist = all([
#     os.path.exists(f"{paths.models_dir}/model_{name}.pkl") 
#     for name in ['logistic', 'random_forest', 'xgboost', 'lightgbm', 'catboost', 'ensemble']
# ])

# if models_exist and not FORCE:
#     print("Models already exist. Loading...")
#     models = {}
#     for name in ['logistic', 'random_forest', 'xgboost', 'lightgbm', 'catboost', 'ensemble']:
#         try:
#             with open(f"{paths.models_dir}/model_{name}.pkl", 'rb') as f:
#                 models[name] = pickle.load(f)
#             print(f"  Loaded {name}")
#         except:
#             pass
# else:
#     # Load tuned parameters
#     tuned_params_path = f"{paths.results_dir}/tuned_params.json"
#     if os.path.exists(tuned_params_path):
#         with open(tuned_params_path, 'r') as f:
#             tuned_params = json.load(f)
#         print("✅ Loaded tuned parameters")
#     else:
#         print("⚠️ No tuned parameters found, using defaults")
#         tuned_params = {}

#     models = {}

#     # 1. Logistic Regression
#     print("\nTraining Logistic Regression with Dask...")
#     try:
#         lr = LogisticRegressionModel(
#             random_state=model_config.random_state,
#             **tuned_params.get('logistic', {})
#         )
#         lr.fit(X_train, y_train)
#         models['logistic'] = lr
#         print("  ✅ Logistic Regression trained")
#     except Exception as e:
#         print(f"  ❌ Logistic Regression failed: {e}")

#     # 2. Random Forest
#     print("\nTraining Random Forest with Dask...")
#     try:
#         rf = RandomForestModel(
#             random_state=model_config.random_state,
#             **tuned_params.get('random_forest', {})
#         )
#         rf.fit(X_train, y_train)
#         models['random_forest'] = rf
#         print("  ✅ Random Forest trained")
#     except Exception as e:
#         print(f"  ❌ Random Forest failed: {e}")

#     # 3. XGBoost
#     print("\nTraining XGBoost with Dask...")
#     try:
#         xgb = XGBoostModel(
#             random_state=model_config.random_state,
#             **tuned_params.get('xgboost', {})
#         )
#         xgb.fit(X_train, y_train, X_val, y_val)
#         models['xgboost'] = xgb
#         print("  ✅ XGBoost trained")
#     except Exception as e:
#         print(f"  ❌ XGBoost failed: {e}")

#     # 4. LightGBM
#     print("\nTraining LightGBM with Dask...")
#     try:
#         lgb = LightGBMModel(
#             random_state=model_config.random_state,
#             **tuned_params.get('lightgbm', {})
#         )
#         lgb.fit(X_train, y_train, X_val, y_val)
#         models['lightgbm'] = lgb
#         print("  ✅ LightGBM trained")
#     except Exception as e:
#         print(f"  ❌ LightGBM failed: {e}")

#     # 5. CatBoost
#     print("\nTraining CatBoost...")
#     try:
#         cat = CatBoostModel(
#             random_state=model_config.random_state,
#             **tuned_params.get('catboost', {})
#         )
#         cat.fit(X_train, y_train, X_val, y_val)
#         models['catboost'] = cat
#         print("  ✅ CatBoost trained")
#     except Exception as e:
#         print(f"  ❌ CatBoost failed: {e}")

#     # 6. Ensemble
#     if len(models) >= 2:
#         print("\nTraining Stacking Ensemble with Dask...")
#         try:
#             ensemble = StackingEnsemble(
#                 base_models=list(models.values()),
#                 meta_model=LightGBMModel(
#                     random_state=model_config.random_state,
#                     is_unbalance=True
#                 ),
#                 cv_folds=min(3, model_config.cv_folds),
#                 random_state=model_config.random_state
#             )
#             ensemble.fit(X_train, y_train, X_val, y_val)
#             models['ensemble'] = ensemble
#             print("  ✅ Ensemble trained")
#         except Exception as e:
#             print(f"  ❌ Ensemble failed: {e}")

#     # Save models
#     print("\nSaving models...")
#     for name, model in models.items():
#         with open(f"{paths.models_dir}/model_{name}.pkl", 'wb') as f:
#             pickle.dump(model, f)
#         print(f"  Saved {name} model")

# print("\n" + "=" * 80)
# print("TRAINING COMPLETED!")
# print("=" * 80)
# print(f"\nModels trained: {len(models)}")
# for name in models.keys():
#     print(f"  - {name}")
# print(f"\nModels saved to: {paths.models_dir}")

# print("\n✅ Module 9 completed.")





MODULE 9: MODEL TRAINING (Chunk-Based)
Encoding categorical variables for model compatibility...


NotImplementedError: `get_dummies` with non-categorical dtypes is not supported. Please use `df.categorize()` beforehand to convert to categorical dtype.

In [ ]:
# =============================================================================
# MODULE 9: MODEL TRAINING
# =============================================================================

print("\n" + "=" * 80)
print("MODULE 9: MODEL TRAINING")
print("=" * 80)

import os
import json
import pickle

from models.logistic import LogisticRegressionModel
from models.random_forest import RandomForestModel
from models.xgboost_model import XGBoostModel
from models.lightgbm_model import LightGBMModel
from models.catboost_model import CatBoostModel
from models.ensemble import StackingEnsemble

# -----------------------------------------------------------------------------
# Load tuned hyperparameters
# -----------------------------------------------------------------------------

tuned_params_path = os.path.join(
    paths.results_dir,
    "tuned_params.json"
)

if os.path.exists(tuned_params_path):

    with open(tuned_params_path, "r") as f:
        tuned_params = json.load(f)

    print("✓ Loaded tuned hyperparameters")

else:

    tuned_params = {}

    print("⚠ No tuned parameters found. Using defaults.")

# -----------------------------------------------------------------------------
# Build model registry
# -----------------------------------------------------------------------------

model_registry = {

    "logistic": LogisticRegressionModel(
        random_state=model_config.random_state,
        **tuned_params.get("logistic", {})
    ),

    "random_forest": RandomForestModel(
        random_state=model_config.random_state,
        **tuned_params.get("random_forest", {})
    ),

    "xgboost": XGBoostModel(
        random_state=model_config.random_state,
        **tuned_params.get("xgboost", {})
    ),

    "lightgbm": LightGBMModel(
        random_state=model_config.random_state,
        **tuned_params.get("lightgbm", {})
    ),

    "catboost": CatBoostModel(
        random_state=model_config.random_state,
        **tuned_params.get("catboost", {})
    )
}

models = {}

# -----------------------------------------------------------------------------
# Train Base Models
# -----------------------------------------------------------------------------

print("\nTraining Base Models")

print("-" * 80)

for name, model in model_registry.items():

    print(f"\n{name.upper()}")

    try:
        print(type(X_train))
        print(type(y_train))

        print(X_train.npartitions)
        print(y_train.npartitions)

        print(X_train.dtypes)
        model.fit(

            X_train=X_train,
            y_train=y_train,

            X_val=X_val,
            y_val=y_val

        )

        models[name] = model

        print("✓ Training completed")

    except Exception as e:

        print(f"✗ Training failed")

        print(e)

# -----------------------------------------------------------------------------
# Train Stacking Ensemble
# -----------------------------------------------------------------------------

if len(models) >= 2:

    print("\nSTACKING ENSEMBLE")

    print("-" * 80)

    try:

        ensemble = StackingEnsemble(

            base_models=models,

            random_state=model_config.random_state,

            cv_folds=min(
                3,
                model_config.cv_folds
            )

        )

        ensemble.fit(

            X_train=X_train,
            y_train=y_train,

            X_val=X_val,
            y_val=y_val

        )

        models["ensemble"] = ensemble

        print("✓ Ensemble trained")

    except Exception as e:

        print("✗ Ensemble training failed")

        print(e)

else:

    print("\nSkipping ensemble (insufficient base models).")

# -----------------------------------------------------------------------------
# Save Models
# -----------------------------------------------------------------------------

print("\nSaving Models")

print("-" * 80)

os.makedirs(paths.models_dir, exist_ok=True)

for name, model in models.items():

    model_path = os.path.join(

        paths.models_dir,

        f"model_{name}.pkl"

    )

    try:

        with open(model_path, "wb") as f:

            pickle.dump(model, f)

        print(f"✓ {name}")

    except Exception as e:

        print(f"✗ Could not save {name}")

        print(e)

# -----------------------------------------------------------------------------
# Summary
# -----------------------------------------------------------------------------

print("\n" + "=" * 80)

print("MODEL TRAINING COMPLETED")

print("=" * 80)

print(f"\nModels Successfully Trained : {len(models)}")

for name in models:

    print(f"   • {name}")

print(f"\nModel Directory : {paths.models_dir}")

print("\n✅ Module 9 completed.")

2026-07-11 02:32:26,419 - models.logistic - INFO - Training Logistic Regression with Dask...



MODULE 9: MODEL TRAINING
✓ Loaded tuned hyperparameters

Training Base Models
--------------------------------------------------------------------------------

LOGISTIC
<class 'dask.dataframe.dask_expr._collection.DataFrame'>
<class 'dask.dataframe.dask_expr._collection.DataFrame'>
32
1
CURRENT_ACTUAL_UPB                            float64
CURRENT_LOAN_DELINQUENCY_STATUS       string[pyarrow]
LOAN_AGE                                      float64
REMAINING_MONTHS_TO_LEGAL_MATURITY            float64
DEFECT_SETTLEMENT_DATE                string[pyarrow]
                                           ...       
future_termination                              int32
is_terminated                                   int32
row_num                                         int32
cumulative_delinquency                          int32
origination_year                                int32
Length: 119, dtype: object


2026-07-11 02:32:39,347 - distributed.nanny - WARNING - Restarting worker (status=Status.running)


In [ ]:
# # =============================================================================
# # MODULE 10: MODEL EVALUATION
# # =============================================================================

# print("\n" + "=" * 80)
# print("MODULE 10: MODEL EVALUATION")
# print("=" * 80)

# from evaluation.metrics import CreditRiskMetrics

# # Load models if not already in memory
# if 'models' not in dir() or not models:
#     print("\nLoading models from disk...")
#     model_names = ['logistic', 'random_forest', 'xgboost', 'lightgbm', 'catboost', 'ensemble']
#     models = {}
#     for name in model_names:
#         try:
#             with open(f"{paths.models_dir}/model_{name}.pkl", 'rb') as f:
#                 models[name] = pickle.load(f)
#             print(f"  Loaded {name}")
#         except:
#             pass

# results = {}
# metrics_calculator = CreditRiskMetrics()

# # Convert y_test to numpy for evaluation
# y_test_np = y_test.compute().values

# print("\nEvaluating models...")

# for name, model in models.items():
#     print(f"\nEvaluating {name}...")
#     try:
#         # Get predictions (returns numpy arrays)
#         y_proba = model.predict_proba(X_test)[:, 1]
#         y_pred = model.predict(X_test)
        
#         # Compute metrics
#         metrics = metrics_calculator.evaluate(y_test_np, y_pred, y_proba)
#         metrics['name'] = name
        
#         # Get feature importance
#         if hasattr(model, 'get_feature_importance'):
#             importance = model.get_feature_importance()
#             if importance:
#                 metrics['feature_importance'] = importance
        
#         # Get lift/gain
#         lift_data = metrics_calculator.compute_lift_gain(y_test_np, y_proba)
#         metrics['lift_data'] = lift_data
        
#         results[name] = metrics
        
#         print(f"  ROC-AUC: {metrics['roc_auc']:.4f}")
#         print(f"  PR-AUC: {metrics['pr_auc']:.4f}")
#         print(f"  F1: {metrics['f1']:.4f}")
#         print(f"  KS: {metrics['ks_statistic']:.4f}")
        
#     except Exception as e:
#         print(f"  ❌ Evaluation failed: {e}")

# # Save results
# if results:
#     results_df = pd.DataFrame([
#         {
#             'model': name,
#             'roc_auc': metrics['roc_auc'],
#             'pr_auc': metrics['pr_auc'],
#             'f1': metrics['f1'],
#             'precision': metrics['precision'],
#             'recall': metrics['recall'],
#             'balanced_accuracy': metrics['balanced_accuracy'],
#             'mcc': metrics['mcc'],
#             'brier_score': metrics['brier_score'],
#             'log_loss': metrics['log_loss'],
#             'ks_statistic': metrics['ks_statistic']
#         }
#         for name, metrics in results.items()
#     ])
#     results_df.to_csv(f"{paths.results_dir}/model_results.csv", index=False)

#     # Summary table
#     print("\n" + "=" * 80)
#     print("MODEL PERFORMANCE SUMMARY")
#     print("=" * 80)
#     print(f"{'Model':<20} {'ROC-AUC':<10} {'PR-AUC':<10} {'F1':<10} {'KS':<10}")
#     print("-" * 80)
#     for _, row in results_df.iterrows():
#         print(f"{row['model']:<20} {row['roc_auc']:<10.4f} {row['pr_auc']:<10.4f} "
#               f"{row['f1']:<10.4f} {row['ks_statistic']:<10.4f}")
#     print("-" * 80)

#     print(f"\nResults saved to: {paths.results_dir}/model_results.csv")

# print("\n✅ Module 10 completed.")

In [ ]:
# # =============================================================================
# # MODULE 11: PROBABILITY CALIBRATION
# # =============================================================================

# print("\n" + "=" * 80)
# print("MODULE 11: PROBABILITY CALIBRATION")
# print("=" * 80)

# from evaluation.calibration import ProbabilityCalibrator

# # Get the best model (ensemble or lightgbm)
# best_model = models.get('ensemble')
# if best_model is None:
#     best_model = models.get('lightgbm')

# if best_model is not None:
#     print(f"\nCalibrating {best_model.name}...")
    
#     calibrator = ProbabilityCalibrator(method='isotonic')
#     calibrated_model = calibrator.calibrate(
#         best_model,
#         X_train, y_train,
#         X_val, y_val
#     )
    
#     # Save calibrated model
#     with open(f"{paths.models_dir}/model_{best_model.name}_calibrated.pkl", 'wb') as f:
#         pickle.dump(calibrated_model, f)
    
#     print(f"  ✅ {best_model.name} calibrated")
#     print(f"Saved {best_model.name}_calibrated model")
# else:
#     print("⚠️ No model available for calibration")

# print("\n✅ Module 11 completed.")

In [ ]:
# # =============================================================================
# # MODULE 12: CREDIT SCORE GENERATION
# # =============================================================================

# print("\n" + "=" * 80)
# print("MODULE 12: CREDIT SCORE GENERATION")
# print("=" * 80)

# from scoring.score_generator import CreditScoreGenerator

# # Get the calibrated ensemble
# try:
#     with open(f"{paths.models_dir}/model_ensemble_calibrated.pkl", 'rb') as f:
#         scoring_model = pickle.load(f)
#     print("\nLoaded calibrated ensemble model")
# except:
#     scoring_model = models.get('ensemble')
#     if scoring_model is None:
#         scoring_model = models.get('lightgbm')
#     print("\nUsing uncalibrated ensemble model")

# if scoring_model is not None:
#     print("Generating credit scores...")
    
#     # Get probabilities
#     y_proba = scoring_model.predict_proba(X_test)[:, 1]
#     y_test_np = y_test.compute().values
    
#     # Create score generator
#     score_generator = CreditScoreGenerator(
#         min_score=300,
#         max_score=900,
#         target_default_rate=0.05,
#         pdo=20
#     )
    
#     # Generate scores
#     score_results = pd.DataFrame({
#         'probability': y_proba,
#         'true_label': y_test_np
#     })
    
#     score_results = score_generator.generate_all_scores(
#         score_results,
#         prob_col='probability',
#         score_col='credit_score'
#     )
    
#     # Score distribution
#     print("\nScore Distribution:")
#     print(score_results['credit_score'].describe())
    
#     # Risk band analysis
#     print("\nRisk Band Analysis:")
#     for band in ['Low', 'Medium', 'High']:
#         band_data = score_results[score_results['risk_band'] == band]
#         if len(band_data) > 0:
#             print(f"  {band}:")
#             print(f"    Observations: {len(band_data):,}")
#             print(f"    Default Rate: {band_data['true_label'].mean():.2%}")
#             print(f"    % of Portfolio: {len(band_data)/len(score_results):.2%}")
    
#     # Save score results
#     score_results.to_csv(f"{paths.results_dir}/scores.csv", index=False)
#     print(f"\nScores saved to: {paths.results_dir}/scores.csv")
# else:
#     print("⚠️ No model available for scoring")

# print("\n✅ Module 12 completed.")

In [ ]:
# # =============================================================================
# # MODULE 13: VISUALIZATION
# # =============================================================================

# print("\n" + "=" * 80)
# print("MODULE 13: VISUALIZATION")
# print("=" * 80)

# from evaluation.plots import CreditRiskVisualizer

# # Create visualizer
# visualizer = CreditRiskVisualizer(save_dir=f"{paths.results_dir}/plots")

# print("\nCreating visualizations...")

# # Check if results exist
# if 'results' in dir() and results:
#     y_test_np = y_test.compute().values
    
#     # Create individual model evaluation reports
#     for name, metrics in results.items():
#         if 'feature_importance' in metrics and metrics['feature_importance']:
#             try:
#                 visualizer.create_model_evaluation_report(
#                     y_test_np,
#                     metrics.get('y_proba'),
#                     metrics.get('y_pred'),
#                     name,
#                     metrics.get('feature_importance'),
#                     save_name=f"{name}_report"
#                 )
#                 print(f"  ✅ Created {name}_report")
#             except Exception as e:
#                 print(f"  ❌ Failed to create {name}_report: {e}")
    
#     # Create ensemble comparison plot
#     try:
#         visualizer.create_ensemble_comparison_plot(
#             results,
#             save_name="ensemble_comparison"
#         )
#         print("  ✅ Created ensemble_comparison")
#     except Exception as e:
#         print(f"  ❌ Failed to create ensemble_comparison: {e}")
# else:
#     print("⚠️ No results available for visualization")

# print(f"\nVisualizations saved to: {paths.results_dir}/plots")

# print("\n✅ Module 13 completed.")

In [ ]:
# # =============================================================================
# # MODULE 14: SHAP EXPLAINABILITY (Optional)
# # =============================================================================

# print("\n" + "=" * 80)
# print("MODULE 14: SHAP EXPLAINABILITY (Optional)")
# print("=" * 80)

# try:
#     import shap
#     print("✅ SHAP is available")
    
#     # Get the best model
#     explain_model = models.get('ensemble') or models.get('lightgbm')
    
#     if explain_model is not None:
#         print(f"\nUsing {explain_model.name} model for SHAP analysis...")
        
#         # Sample data for SHAP (5,000 records)
#         print("\nSampling 5,000 records for SHAP analysis...")
#         X_sample = X_test.head(5000).compute()
        
#         # Create SHAP explainer based on model type
#         if hasattr(explain_model, 'model'):
#             if hasattr(explain_model.model, 'get_booster'):
#                 explainer = shap.TreeExplainer(explain_model.model)
#             elif hasattr(explain_model.model, 'booster_'):
#                 explainer = shap.TreeExplainer(explain_model.model)
#             else:
#                 print("Using KernelExplainer (sampling 100 background points)...")
#                 X_background = X_sample.sample(100, random_state=42)
#                 explainer = shap.KernelExplainer(
#                     explain_model.predict_proba,
#                     X_background
#                 )
#         else:
#             X_background = X_sample.sample(100, random_state=42)
#             explainer = shap.KernelExplainer(
#                 explain_model.predict_proba,
#                 X_background
#             )
        
#         # Calculate SHAP values
#         print("Calculating SHAP values...")
#         shap_values = explainer.shap_values(X_sample)
        
#         # Create summary plot
#         plt.figure(figsize=(12, 8))
#         shap.summary_plot(shap_values, X_sample, show=False)
#         plt.tight_layout()
#         plt.savefig(f"{paths.results_dir}/plots/shap_summary.png", dpi=300, bbox_inches='tight')
#         plt.close()
        
#         print("SHAP analysis completed.")
#         print(f"SHAP plot saved to: {paths.results_dir}/plots/shap_summary.png")
#     else:
#         print("⚠️ No model available for SHAP analysis")
        
# except ImportError:
#     print("⚠️ SHAP not installed. Install with: pip install shap")
# except Exception as e:
#     print(f"⚠️ SHAP analysis failed: {e}")

# print("\n✅ Module 14 completed.")

In [ ]:
# # =============================================================================
# # MODULE 15: CLEANUP
# # =============================================================================

# print("\n" + "=" * 80)
# print("MODULE 15: CLEANUP")
# print("=" * 80)

# # Close Dask client
# if 'dask_client' in dir() and dask_client:
#     print("\nClosing Dask client...")
#     dask_client.close()

# # Stop Spark session
# print("Stopping Spark session...")
# spark.stop()

# print("\n" + "=" * 80)
# print("NOTEBOOK COMPLETED SUCCESSFULLY!")
# print("=" * 80)
# print(f"\nResults saved to: {paths.results_dir}")
# print(f"Models saved to: {paths.models_dir}")
# print(f"Data saved to: {paths.data_dir}")

# print("\n✅ Module 15 completed.")

2026-07-11 02:09:08,815 - distributed.nanny - WARNING - Restarting worker (status=Status.running)
